# 🇪🇺 EU AI Act Risk Classifier
**Author:** Olivia Athelus, M.S. | AIxGRC  
**GitHub:** [github.com/aixgrc](https://github.com/aixgrc)  
**LinkedIn:** [linkedin.com/in/oliviaathelus](https://linkedin.com/in/oliviaathelus)

---

## Overview

This notebook implements a rule-based classifier that maps an AI use case to its **risk classification under the EU Artificial Intelligence Act (EU AI Act)**.

The EU AI Act (Regulation EU 2024/1689) applies a tiered risk-based approach to AI systems:

| Risk Level | Description |
|---|---|
| 🔴 **Unacceptable Risk** | Prohibited AI practices — banned outright |
| 🟠 **High Risk** | Permitted but subject to strict requirements |
| 🟡 **Limited Risk** | Transparency obligations apply |
| 🟢 **Minimal Risk** | No specific obligations — freely permitted |

---

## How to Use

1. Run all cells in order
2. When prompted, describe your AI use case in plain English
3. The classifier will output:
   - Risk classification
   - Applicable EU AI Act article references
   - Compliance obligations
   - Recommended next steps

---

> **Disclaimer:** This tool is for educational and research purposes. It does not constitute legal advice. Always consult qualified legal counsel for formal compliance determinations.

In [ ]:
# ─────────────────────────────────────────────
# EU AI Act Risk Classifier
# AIxGRC | Olivia Athelus
# ─────────────────────────────────────────────

import re

# ── Risk Category Definitions ──

UNACCEPTABLE_KEYWORDS = [
    # Social scoring
    "social scoring", "citizen scoring", "social credit",
    # Subliminal manipulation
    "subliminal", "manipulate behavior", "exploit vulnerability",
    "exploit weakness", "unconscious manipulation",
    # Real-time biometric surveillance
    "real-time biometric", "real time biometric", "live facial recognition",
    "mass surveillance", "public surveillance biometric",
    # Emotion inference in workplace/education
    "emotion recognition workplace", "emotion recognition school",
    "emotion recognition education",
    # Predictive policing based on profiling
    "predictive policing", "predict crime based on profile",
    # Biometric categorisation from sensitive traits
    "biometric categorization race", "biometric categorization religion",
    "biometric categorisation political",
]

HIGH_RISK_KEYWORDS = [
    # Critical infrastructure
    "critical infrastructure", "power grid", "water supply", "energy network",
    "transport safety", "traffic management",
    # Education & vocational
    "student assessment", "educational assessment", "exam scoring",
    "admission decision", "evaluate students", "academic evaluation",
    # Employment
    "hiring", "recruitment", "job application screening", "cv screening",
    "employee monitoring", "performance evaluation", "termination decision",
    "workforce management",
    # Essential services
    "credit scoring", "credit decision", "loan application", "insurance risk",
    "benefit eligibility", "social benefit", "welfare eligibility",
    "creditworthiness",
    # Law enforcement
    "law enforcement", "criminal investigation", "evidence assessment",
    "risk assessment criminal", "lie detection", "polygraph",
    "border control", "migration assessment", "asylum",
    # Administration of justice
    "judicial decision", "court decision", "legal ruling", "sentencing",
    # Medical
    "medical device", "clinical decision", "diagnosis ai", "ai diagnosis",
    "surgical robot", "patient triage", "medical imaging ai",
    # Biometric identification (non-real-time)
    "biometric identification", "face recognition database",
    "remote biometric", "fingerprint identification",
]

LIMITED_RISK_KEYWORDS = [
    "chatbot", "chat bot", "virtual assistant", "conversational ai",
    "customer service bot", "ai generated", "deepfake", "synthetic media",
    "generated image", "generated video", "generated audio",
    "ai content", "emotion detection", "sentiment analysis customer",
    "recommendation engine", "personalization", "targeted advertising",
]

print(" EU AI Act keyword rules loaded successfully.")
print(f"   Unacceptable risk patterns : {len(UNACCEPTABLE_KEYWORDS)}")
print(f"   High risk patterns         : {len(HIGH_RISK_KEYWORDS)}")
print(f"   Limited risk patterns      : {len(LIMITED_RISK_KEYWORDS)}")

In [ ]:
# ─────────────────────────────────────────────
# Compliance Obligations by Risk Level
# ─────────────────────────────────────────────

OBLIGATIONS = {
    "UNACCEPTABLE": {
        "level": " UNACCEPTABLE RISK",
        "verdict": "PROHIBITED under EU AI Act",
        "articles": "Article 5 — Prohibited AI Practices",
        "obligations": [
            "This AI system or practice is BANNED under the EU AI Act.",
            "Deployment is illegal in the European Union.",
            "No exemptions apply to this category.",
            "Non-compliance may result in fines up to €35 million or 7% of global annual turnover.",
        ],
        "next_steps": [
            "Do not deploy this system in the EU.",
            "Consult legal counsel immediately if system is already deployed.",
            "Review Article 5 of the EU AI Act for the full list of prohibited practices.",
            "Redesign the use case to remove prohibited functionality.",
        ]
    },
    "HIGH": {
        "level": " HIGH RISK",
        "verdict": "PERMITTED with strict compliance obligations",
        "articles": "Articles 6–49 — High-Risk AI Systems",
        "obligations": [
            "Establish and maintain a risk management system (Article 9).",
            "Implement data governance and training data quality controls (Article 10).",
            "Prepare and maintain technical documentation (Article 11).",
            "Ensure automatic logging and record-keeping (Article 12).",
            "Provide transparency and instructions for use (Article 13).",
            "Ensure human oversight mechanisms are in place (Article 14).",
            "Meet accuracy, robustness, and cybersecurity requirements (Article 15).",
            "Register system in the EU AI Act public database (Article 49).",
            "Conduct conformity assessment before deployment (Articles 43–48).",
        ],
        "next_steps": [
            "Conduct a formal AI risk assessment aligned to NIST AI RMF or ISO 42001.",
            "Appoint an AI compliance officer or responsible person.",
            "Prepare technical documentation package.",
            "Implement bias testing and accuracy validation.",
            "Register in the EU high-risk AI database.",
            "Engage a notified body for conformity assessment if required.",
        ]
    },
    "LIMITED": {
        "level": " LIMITED RISK",
        "verdict": "PERMITTED with transparency obligations",
        "articles": "Articles 50–52 — Transparency Obligations",
        "obligations": [
            "Disclose to users that they are interacting with an AI system (Article 50).",
            "Label AI-generated content (deepfakes, synthetic media) appropriately.",
            "Ensure chatbots identify themselves as AI when directly asked.",
            "Apply transparency measures for emotion recognition systems.",
        ],
        "next_steps": [
            "Add clear AI disclosure notices to user interfaces.",
            "Implement AI content labelling for generated media.",
            "Review user communication flows for transparency compliance.",
            "Document transparency measures for audit readiness.",
        ]
    },
    "MINIMAL": {
        "level": " MINIMAL RISK",
        "verdict": "PERMITTED — no specific EU AI Act obligations",
        "articles": "Article 53 — General-Purpose AI (GPAI) provisions may apply",
        "obligations": [
            "No mandatory requirements under the EU AI Act for this category.",
            "Voluntary codes of conduct are encouraged (Article 95).",
            "General data protection obligations (GDPR) still apply.",
            "Monitor for reclassification if use case evolves.",
        ],
        "next_steps": [
            "Adopt voluntary AI ethics guidelines and responsible AI principles.",
            "Ensure GDPR compliance for any personal data processed.",
            "Document AI use cases for governance records.",
            "Reassess classification if system scope or use case changes.",
        ]
    }
}

print(" Compliance obligations framework loaded.")

In [ ]:
# ─────────────────────────────────────────────
# Classifier Function
# ─────────────────────────────────────────────

def classify_eu_ai_act(use_case: str) -> dict:
    """
    Classifies an AI use case under the EU AI Act risk framework.
    
    Parameters:
        use_case (str): Plain-language description of the AI system or use case.
    
    Returns:
        dict: Classification result with risk level, obligations, and next steps.
    """
    text = use_case.lower()
    matched_keywords = []

    # Check Unacceptable Risk first (highest priority)
    for kw in UNACCEPTABLE_KEYWORDS:
        if kw in text:
            matched_keywords.append(kw)
    if matched_keywords:
        result = OBLIGATIONS["UNACCEPTABLE"].copy()
        result["matched_keywords"] = matched_keywords
        result["use_case"] = use_case
        return result

    # Check High Risk
    for kw in HIGH_RISK_KEYWORDS:
        if kw in text:
            matched_keywords.append(kw)
    if matched_keywords:
        result = OBLIGATIONS["HIGH"].copy()
        result["matched_keywords"] = matched_keywords
        result["use_case"] = use_case
        return result

    # Check Limited Risk
    for kw in LIMITED_RISK_KEYWORDS:
        if kw in text:
            matched_keywords.append(kw)
    if matched_keywords:
        result = OBLIGATIONS["LIMITED"].copy()
        result["matched_keywords"] = matched_keywords
        result["use_case"] = use_case
        return result

    # Default: Minimal Risk
    result = OBLIGATIONS["MINIMAL"].copy()
    result["matched_keywords"] = []
    result["use_case"] = use_case
    return result


def print_classification(result: dict):
    """Pretty-prints a classification result."""
    print("\n" + "═" * 60)
    print(f"  EU AI ACT RISK CLASSIFICATION REPORT")
    print("═" * 60)
    print(f"\n USE CASE:")
    print(f"   {result['use_case']}")
    print(f"\n️  RISK LEVEL:")
    print(f"   {result['level']}")
    print(f"\n VERDICT:")
    print(f"   {result['verdict']}")
    print(f"\n APPLICABLE ARTICLES:")
    print(f"   {result['articles']}")
    if result['matched_keywords']:
        print(f"\n MATCHED INDICATORS:")
        for kw in result['matched_keywords']:
            print(f"   • {kw}")
    print(f"\n COMPLIANCE OBLIGATIONS:")
    for ob in result['obligations']:
        print(f"   • {ob}")
    print(f"\n️  RECOMMENDED NEXT STEPS:")
    for step in result['next_steps']:
        print(f"   {result['next_steps'].index(step)+1}. {step}")
    print("\n" + "═" * 60)


print(" Classifier ready. Run the next cell to classify a use case.")

## 🔎 Classify Your AI Use Case

Run the cell below and enter a description of your AI system when prompted.  
Or replace the `use_case` variable with your own text to run without the prompt.

In [ ]:
# ─────────────────────────────────────────────
# Interactive Classification
# ─────────────────────────────────────────────

# Option 1: Interactive input
use_case = input("\nDescribe your AI use case: ")

# Option 2: Uncomment and edit the line below to hardcode a use case
# use_case = "An AI system that screens job applications and ranks candidates for hiring decisions"

result = classify_eu_ai_act(use_case)
print_classification(result)

## 📊 Batch Classification — Test Multiple Use Cases

Run the cell below to classify a set of example use cases and see results side by side.

In [ ]:
# ─────────────────────────────────────────────
# Batch Classification with Example Use Cases
# ─────────────────────────────────────────────

test_cases = [
    "A government system that assigns social scoring to citizens based on behavior to restrict access to services",
    "An AI tool used for CV screening and hiring decisions at a large corporation",
    "A credit scoring model used by a bank to assess loan applications",
    "A chatbot used on a retail website to answer customer service questions",
    "An AI model that generates personalized product recommendations for an e-commerce platform",
    "A real-time biometric surveillance system deployed in public spaces by law enforcement",
    "A spam filter that automatically classifies and removes unwanted emails",
    "An AI system that assists judges in determining criminal sentencing recommendations",
    "A medical imaging AI that helps radiologists detect tumors in scan results",
    "A deepfake video generation tool for entertainment and creative media",
]

print("\n" + "═" * 60)
print("  BATCH CLASSIFICATION RESULTS")
print("═" * 60)
print(f"{'#':<4} {'Risk Level':<25} {'Use Case (truncated)':<40}")
print("-" * 70)

for i, case in enumerate(test_cases, 1):
    result = classify_eu_ai_act(case)
    truncated = case[:55] + "..." if len(case) > 55 else case
    print(f"{i:<4} {result['level']:<30} {truncated}")

print("\n" + "═" * 60)
print("\nRun print_classification(classify_eu_ai_act(case)) on any")
print("case above for the full compliance report.")

## 📚 EU AI Act Reference Guide

### Risk Tier Summary

| Tier | Examples | Max Fine |
|---|---|---|
| 🔴 Unacceptable | Social scoring, real-time mass biometric surveillance, subliminal manipulation | €35M or 7% global turnover |
| 🟠 High Risk | Hiring AI, credit scoring, medical devices, law enforcement tools, border control | €15M or 3% global turnover |
| 🟡 Limited Risk | Chatbots, deepfakes, emotion recognition, recommendation engines | €7.5M or 1.5% global turnover |
| 🟢 Minimal Risk | Spam filters, AI in games, basic automation | No mandatory obligations |

### Key Articles

| Article | Topic |
|---|---|
| Article 5 | Prohibited AI Practices |
| Article 6 | Classification of High-Risk AI Systems |
| Article 9 | Risk Management System |
| Article 10 | Data and Data Governance |
| Article 11 | Technical Documentation |
| Article 13 | Transparency and Information |
| Article 14 | Human Oversight |
| Article 50 | Transparency for Certain AI Systems |
| Article 95 | Codes of Conduct |

---

### About This Notebook

Built by **Olivia Athelus** as part of the **AIxGRC** project — a personal research portfolio at the intersection of AI governance, risk management, and compliance.  

🌐 [aixgrc.github.io](https://aixgrc.github.io)  
🔗 [linkedin.com/in/oliviaathelus](https://linkedin.com/in/oliviaathelus)  
💻 [github.com/aixgrc](https://github.com/aixgrc)

> This classifier uses keyword-based rule matching and is intended for educational use. For production compliance determinations, engage qualified legal and compliance professionals.